# Hierarchical GeoNames locality resolution

This experiment resolves a locality from broad administrative context to narrower features. GeoNames remains authoritative: names are retrieved from PostgreSQL, administrative codes define hierarchy, and envelopes derived from GeoNames descendants validate that narrower candidates fall inside the resolved context.

A local LLM is used only when a locality component has no hierarchy-compatible candidate. It may propose alternative search terms, but it cannot provide coordinates or identifiers.

Metrics use PlaceBot's pinned Bombus UK reference data. The default reproducible sample is 100 records; set `SAMPLE_SIZE = None` to run all 709.

In [ ]:
from __future__ import annotations

import json
import re
from functools import lru_cache

import numpy as np
import pandas as pd
import psycopg2
import requests
from tqdm.auto import tqdm

from pkb.config import settings
from pkb.utils import country_string_to_iso


In [ ]:
PLACEBOT_COMMIT = "b036dd0b555d5ff32f7775bc0461707cc0df6285"
BENCHMARK_URL = (
    "https://raw.githubusercontent.com/JackDanHollister/PlaceBot/"
    f"{PLACEBOT_COMMIT}/benchmarks/data/bombus_uk_reference.tsv"
)
SAMPLE_SIZE = 100
RANDOM_STATE = 42

all_benchmarks = pd.read_csv(BENCHMARK_URL, sep="\t")
benchmarks = (
    all_benchmarks
    if SAMPLE_SIZE is None
    else all_benchmarks.sample(SAMPLE_SIZE, random_state=RANDOM_STATE)
)
benchmarks = benchmarks.reset_index(drop=True)
benchmarks.shape


## Method

1. Split the verbatim locality into possible place components.
2. Retrieve exact or trigram GeoNames candidates for every component.
3. Identify the strongest administrative candidate and use its GeoNames administrative codes as context.
4. Reject narrower candidates that conflict with that hierarchy or fall outside its descendant envelope.
5. Ask the local model for alternative search terms only for unresolved components.
6. Rank the remaining candidates deterministically and retain unresolved text explicitly.

For this historical UK dataset, the search scope includes GB, IE and IM. An explicit country name in the locality is preferred over the supplied country field.

In [ ]:
connection = psycopg2.connect(
    host=settings.postgres_host,
    port=settings.postgres_port,
    dbname=settings.postgres_database,
    user=settings.postgres_user,
    password=settings.postgres_password,
)

SEARCH_COUNTRIES = ("GB", "IE", "IM")

with connection.cursor() as cursor:
    cursor.execute(
        """
        SELECT geoname_id, latitude, longitude, country_code,
               admin1_code, admin2_code, admin3_code, admin4_code
        FROM places_geonames
        WHERE country_code = ANY(%s)
        """,
        (list(SEARCH_COUNTRIES),),
    )
    geography = pd.DataFrame(
        cursor.fetchall(),
        columns=[
            "geoname_id",
            "latitude",
            "longitude",
            "country_code",
            "admin1_code",
            "admin2_code",
            "admin3_code",
            "admin4_code",
        ],
    )

geography.shape


In [ ]:
ADMIN_COLUMNS = ["admin1_code", "admin2_code", "admin3_code", "admin4_code"]


def build_bounds(frame):
    bounds = {}
    for level in range(1, 5):
        keys = ["country_code", *ADMIN_COLUMNS[:level]]
        grouped = (
            frame.dropna(subset=keys)
            .groupby(keys, dropna=False)
            .agg(
                min_latitude=("latitude", "min"),
                max_latitude=("latitude", "max"),
                min_longitude=("longitude", "min"),
                max_longitude=("longitude", "max"),
                descendant_count=("geoname_id", "size"),
            )
            .reset_index()
        )
        for row in grouped.itertuples(index=False):
            key = tuple(getattr(row, column) for column in keys)
            bounds[(level, key)] = {
                "min_latitude": row.min_latitude,
                "max_latitude": row.max_latitude,
                "min_longitude": row.min_longitude,
                "max_longitude": row.max_longitude,
                "descendant_count": row.descendant_count,
            }
    return bounds


admin_bounds = build_bounds(geography)
len(admin_bounds)


In [ ]:
LABEL_PREFIX = re.compile(r"^\s*\d+(?:st|nd|rd|th)\s+label\s*:\s*", re.IGNORECASE)
DISTANCE_PREFIX = re.compile(
    r"^\s*\d+(?:\.\d+)?\s*(?:km|kms?|m|mi|miles?)\.?\s+"
    r"(?:north|south|east|west|n|s|e|w|ne|nw|se|sw)\.?\s+(?:of\s+)?",
    re.IGNORECASE,
)


def locality_components(locality):
    components = []
    explicit_countries = []
    for value in re.split(r"[,;:]", str(locality or "")):
        value = LABEL_PREFIX.sub("", value).strip(" .")
        value = DISTANCE_PREFIX.sub("", value).strip(" .")
        if len(value) < 3:
            continue
        if country_code := country_string_to_iso(value):
            explicit_countries.append(country_code)
        else:
            components.append(value)
    return list(dict.fromkeys(components)), list(dict.fromkeys(explicit_countries))


def country_scope(row, explicit_countries):
    supplied = country_string_to_iso(str(row.Country))
    if explicit_countries:
        return tuple(explicit_countries)
    if supplied == "GB":
        return SEARCH_COUNTRIES
    return (supplied,) if supplied else SEARCH_COUNTRIES


In [ ]:
SEARCH_QUERY = """
    WITH matches AS (
        SELECT
            search.geoname_id,
            search.canonical_name,
            search.name AS matched_name,
            search.name_type,
            search.is_preferred,
            search.country_code,
            search.feature_class,
            search.feature_code,
            search.latitude,
            search.longitude,
            search.population,
            place.admin1_code,
            place.admin2_code,
            place.admin3_code,
            place.admin4_code,
            search.search_name = %(name)s AS exact_match,
            similarity(search.search_name, %(name)s) AS name_score
        FROM places_geonames_search AS search
        JOIN places_geonames AS place USING (geoname_id)
        WHERE {predicate}
          AND search.country_code = ANY(%(country_codes)s)
    ),
    ranked AS (
        SELECT *,
               row_number() OVER (
                   PARTITION BY geoname_id
                   ORDER BY exact_match DESC,
                            name_score DESC,
                            is_preferred DESC,
                            population DESC NULLS LAST
               ) AS place_rank
        FROM matches
    )
    SELECT *
    FROM ranked
    WHERE place_rank = 1
    ORDER BY exact_match DESC,
             name_score DESC,
             is_preferred DESC,
             population DESC NULLS LAST,
             geoname_id
    LIMIT %(limit)s
"""


@lru_cache(maxsize=None)
def search_component(component, country_codes, limit=10):
    parameters = {
        "name": component.strip().lower(),
        "country_codes": list(country_codes),
        "limit": limit,
    }
    with connection.cursor() as cursor:
        cursor.execute(
            SEARCH_QUERY.format(predicate="search.search_name = %(name)s"),
            parameters,
        )
        rows = cursor.fetchall()
        if not rows:
            cursor.execute("SET LOCAL pg_trgm.similarity_threshold = 0.6")
            cursor.execute(
                SEARCH_QUERY.format(predicate="search.search_name %% %(name)s"),
                parameters,
            )
            rows = cursor.fetchall()
        columns = [column.name for column in cursor.description]
    return [dict(zip(columns, row)) for row in rows]


In [ ]:
ADMIN_LEVELS = {
    "ADM1": 1,
    "ADM2": 2,
    "ADM3": 3,
    "ADM4": 4,
}


def admin_level(candidate):
    return ADMIN_LEVELS.get(candidate["feature_code"], 0)


def context_key(candidate):
    level = admin_level(candidate)
    if not level:
        return None
    values = [candidate["country_code"]]
    values.extend(candidate[column] for column in ADMIN_COLUMNS[:level])
    if any(value is None for value in values):
        return None
    return level, tuple(values)


def choose_context(candidate_sets):
    administrative = [
        candidate
        for candidates in candidate_sets.values()
        for candidate in candidates
        if admin_level(candidate)
        and (candidate["exact_match"] or candidate["name_score"] >= 0.8)
    ]
    if not administrative:
        return None
    return max(
        administrative,
        key=lambda candidate: (
            candidate["exact_match"],
            candidate["name_score"],
            admin_level(candidate),
            candidate["is_preferred"],
            candidate["population"] or 0,
        ),
    )


def within_context(candidate, context):
    if context is None:
        return True
    level = admin_level(context)
    if candidate["country_code"] != context["country_code"]:
        return False
    for column in ADMIN_COLUMNS[:level]:
        expected = context[column]
        observed = candidate[column]
        if expected is not None and observed is not None and expected != observed:
            return False

    bounds = admin_bounds.get(context_key(context))
    if not bounds:
        return True
    margin = 0.05
    return (
        bounds["min_latitude"] - margin
        <= candidate["latitude"]
        <= bounds["max_latitude"] + margin
        and bounds["min_longitude"] - margin
        <= candidate["longitude"]
        <= bounds["max_longitude"] + margin
    )


In [ ]:
OLLAMA_URL = "http://127.0.0.1:11434"
OLLAMA_MODEL = "llama3.1:latest"

try:
    OLLAMA_AVAILABLE = requests.get(f"{OLLAMA_URL}/api/tags", timeout=2).ok
except requests.RequestException:
    OLLAMA_AVAILABLE = False

print(f"Ollama available: {OLLAMA_AVAILABLE}")


@lru_cache(maxsize=None)
def llm_search_terms(component, locality, context_name, bounds_text):
    response = requests.post(
        f"{OLLAMA_URL}/api/generate",
        json={
            "model": OLLAMA_MODEL,
            "stream": False,
            "format": "json",
            "prompt": (
                "Suggest up to three gazetteer search terms for an unresolved "
                "component of a museum locality. Expand abbreviations or likely "
                "OCR errors, but do not invent coordinates or identifiers. Return "
                "JSON with only a search_terms array; use an empty array when the "
                "component is not a place.\n"
                f"Full locality: {locality}\n"
                f"Unresolved component: {component}\n"
                f"Resolved context: {context_name or 'country only'}\n"
                f"Context bounds: {bounds_text or 'unknown'}"
            ),
            "options": {
                "temperature": 0,
                "seed": 0,
                "num_predict": 100,
            },
        },
        timeout=180,
    )
    response.raise_for_status()
    parsed = json.loads(response.json()["response"])
    terms = [
        str(term).strip()
        for term in parsed.get("search_terms", [])
        if len(str(term).strip()) >= 3
    ]
    return list(dict.fromkeys(terms))[:3]


In [ ]:
def context_description(context):
    if context is None:
        return None, None
    bounds = admin_bounds.get(context_key(context))
    bounds_text = (
        json.dumps({key: round(value, 4) for key, value in bounds.items() if key != "descendant_count"})
        if bounds
        else None
    )
    return context["canonical_name"], bounds_text


def candidate_score(candidate, context):
    score = 4 * candidate["exact_match"] + 2 * candidate["name_score"]
    score += 0.3 * candidate["is_preferred"]
    score += 2 if context and within_context(candidate, context) else 0
    score += 1.5 if candidate["feature_class"] != "A" else 0
    score += min(np.log10((candidate["population"] or 0) + 1) / 20, 0.3)
    return score


def resolve_record(row, use_llm):
    components, explicit_countries = locality_components(row.label_verbatim)
    countries = country_scope(row, explicit_countries)
    initial = {
        component: search_component(component, countries)
        for component in components
    }
    context = choose_context(initial)

    candidates_by_component = {
        component: [
            candidate
            for candidate in candidates
            if within_context(candidate, context)
        ]
        for component, candidates in initial.items()
    }

    llm_used = False
    llm_recovered = 0
    context_name, bounds_text = context_description(context)
    if use_llm and OLLAMA_AVAILABLE:
        for component, candidates in candidates_by_component.items():
            if candidates:
                continue
            llm_used = True
            recovered = {}
            for term in llm_search_terms(
                component,
                row.label_verbatim,
                context_name,
                bounds_text,
            ):
                for candidate in search_component(term, countries):
                    if within_context(candidate, context):
                        candidate = {
                            **candidate,
                            "query_component": component,
                            "llm_search_term": term,
                        }
                        recovered[candidate["geoname_id"]] = candidate
            candidates_by_component[component] = list(recovered.values())
            llm_recovered += bool(recovered)

    ranked = {}
    for component, candidates in candidates_by_component.items():
        for candidate in candidates:
            candidate = {
                **candidate,
                "query_component": candidate.get("query_component", component),
            }
            candidate["resolution_score"] = candidate_score(candidate, context)
            current = ranked.get(candidate["geoname_id"])
            if current is None or candidate["resolution_score"] > current["resolution_score"]:
                ranked[candidate["geoname_id"]] = candidate

    ranked = sorted(
        ranked.values(),
        key=lambda candidate: (
            candidate["resolution_score"],
            candidate["exact_match"],
            candidate["name_score"],
        ),
        reverse=True,
    )
    unresolved = [
        component
        for component, candidates in candidates_by_component.items()
        if not candidates
    ]
    return {
        "candidates": ranked[:5],
        "context": context,
        "unresolved_components": unresolved,
        "llm_used": llm_used,
        "llm_recovered_components": llm_recovered,
    }


In [ ]:
def haversine_km(latitude, longitude, reference_latitude, reference_longitude):
    latitude, longitude, reference_latitude, reference_longitude = np.radians(
        [latitude, longitude, reference_latitude, reference_longitude]
    )
    delta_latitude = reference_latitude - latitude
    delta_longitude = reference_longitude - longitude
    value = (
        np.sin(delta_latitude / 2) ** 2
        + np.cos(latitude)
        * np.cos(reference_latitude)
        * np.sin(delta_longitude / 2) ** 2
    )
    return 6371.0088 * 2 * np.arctan2(np.sqrt(value), np.sqrt(1 - value))


def evaluate_resolution(row, resolution, prefix):
    candidates = resolution["candidates"]
    distances = [
        haversine_km(
            candidate["latitude"],
            candidate["longitude"],
            row.lat_manual,
            row.long_manual,
        )
        for candidate in candidates
    ]
    context = resolution["context"]
    return {
        f"{prefix}_candidate_count": len(candidates),
        f"{prefix}_top1_geoname_id": candidates[0]["geoname_id"] if candidates else None,
        f"{prefix}_top1_name": candidates[0]["canonical_name"] if candidates else None,
        f"{prefix}_top1_distance_km": distances[0] if distances else np.nan,
        f"{prefix}_top1_within_radius": bool(
            distances and distances[0] <= row.radius_km_manual
        ),
        f"{prefix}_top5_within_radius": any(
            distance <= row.radius_km_manual for distance in distances
        ),
        f"{prefix}_context": context["canonical_name"] if context else None,
        f"{prefix}_unresolved_components": resolution["unresolved_components"],
        f"{prefix}_llm_used": resolution["llm_used"],
        f"{prefix}_llm_recovered_components": resolution["llm_recovered_components"],
    }


In [ ]:
evaluations = []
for row in tqdm(benchmarks.itertuples(index=False), total=len(benchmarks)):
    hierarchy = resolve_record(row, use_llm=False)
    hierarchy_llm = resolve_record(row, use_llm=True)
    evaluations.append(
        {
            **evaluate_resolution(row, hierarchy, "hierarchy"),
            **evaluate_resolution(row, hierarchy_llm, "hierarchy_llm"),
        }
    )

results = pd.concat(
    [benchmarks.reset_index(drop=True), pd.DataFrame(evaluations)],
    axis=1,
)


In [ ]:
def metrics_for(prefix):
    resolved = results[f"{prefix}_candidate_count"].gt(0)
    correct = results[f"{prefix}_top1_within_radius"]
    return pd.Series(
        {
            "records": len(results),
            "resolved": resolved.sum(),
            "coverage": resolved.mean(),
            "top1_accuracy": correct.mean(),
            "top1_precision": correct.sum() / resolved.sum(),
            "top5_accuracy": results[f"{prefix}_top5_within_radius"].mean(),
            "median_top1_distance_km": results[
                f"{prefix}_top1_distance_km"
            ].median(),
            "components_unresolved": results[
                f"{prefix}_unresolved_components"
            ].map(len).sum(),
            "llm_records": results[f"{prefix}_llm_used"].sum(),
            "llm_components_recovered": results[
                f"{prefix}_llm_recovered_components"
            ].sum(),
        },
        name=prefix,
    )


metrics = pd.concat(
    [
        metrics_for("hierarchy"),
        metrics_for("hierarchy_llm"),
    ],
    axis=1,
)
metrics


In [ ]:
results[
    [
        "Barcode",
        "label_verbatim",
        "radius_km_manual",
        "hierarchy_context",
        "hierarchy_llm_top1_name",
        "hierarchy_llm_top1_distance_km",
        "hierarchy_llm_top1_within_radius",
        "hierarchy_llm_unresolved_components",
        "hierarchy_llm_llm_used",
    ]
].sort_values(
    ["hierarchy_llm_top1_within_radius", "hierarchy_llm_top1_distance_km"],
    ascending=[True, False],
).head(30)


In [ ]:
connection.close()
